# TürkResearcher — Stage 2a: Continued Pre-Training (RESUME-READY)

Trendyol-LLM-7b-base-v1.0 üzerinde 387M token Türkçe akademik continued pre-train. Checkpoint Drive'a kaydeder, hard kill'lerden kurtulur.

**Önce:** Runtime → Change runtime type → **A100 GPU** (Colab Pro+).

**Çalıştırma sırası:**
1. CUDA env (ZORUNLU İLK)
2. Install (pinned versions)
3. Workspace + HF auth
4. Mount Drive (persistent checkpoints)
5. Pull pre-tokenized corpus
6. Build HF Datasets
7. Load Trendyol-7B base
8. torch.load weights_only patch
9. Full training (Drive'dan resume otomatik)
10. Final val PPL
11. Sample generations
12. Push to HF Hub

**Recipe:** LR=1e-5, effective batch=32, bf16 + paged_adamw_8bit + gradient_checkpointing, max_seq=2048, 1 epoch, save_steps=1000 (Drive).

## 1. CUDA env (ZORUNLU İLK CELL)

`expandable_segments` PyTorch fragmentation'ı önler. `import torch`'tan önce set edilmek zorunda.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('CUDA alloc conf:', os.environ.get('PYTORCH_CUDA_ALLOC_CONF'))

## 2. Install (pinned versions)

- transformers 4.45.2: 4.46+ Trendyol-LLM-7b reposunda 404 fırlatıyor
- bitsandbytes: 8-bit paged AdamW için

In [ ]:
!nvidia-smi
!pip install -q --force-reinstall "transformers==4.45.2" "tokenizers>=0.20,<0.21" "datasets>=2.20" "accelerate>=0.34" sentencepiece huggingface_hub pyarrow bitsandbytes

## 3. Workspace + HF auth

In [ ]:
import os
WORK_DIR = '/content/trakad_pretrain'
os.makedirs(WORK_DIR, exist_ok=True)
print('Working dir:', WORK_DIR)
!df -h /content | tail -1

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('HF write token: ')
login(hf_token, add_to_git_credential=False)

## 4. Mount Google Drive (persistent checkpoint storage)

Checkpoint'ler Drive'a kaydedilir (persistent). `/content` ephemeral — hard kill'de silinir.

İlk runda Drive klasörü boş olur, training sıfırdan başlar. Tekrar runlarda Drive'daki son checkpoint'ten resume eder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/trakad-7b-base'
os.makedirs(DRIVE_OUT, exist_ok=True)
ckpts = sorted([d for d in os.listdir(DRIVE_OUT) if d.startswith('checkpoint-')])
print(f'Drive checkpoints found: {ckpts if ckpts else "none — will start fresh"}')

## 5. Pull pre-tokenized corpus from HF Hub

Source: `hakansabunis/tr-academic-research-agent-index/pretrain_corpus/` — 222,830 chunks of max_seq=2048.

In [ ]:
from huggingface_hub import hf_hub_download
import json

TRAIN_PARQUET = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='pretrain_corpus/train.parquet',
    repo_type='dataset',
    local_dir=WORK_DIR,
)
VAL_PARQUET = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='pretrain_corpus/val.parquet',
    repo_type='dataset',
    local_dir=WORK_DIR,
)
STATS = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='pretrain_corpus/_corpus_stats.json',
    repo_type='dataset',
    local_dir=WORK_DIR,
)

stats = json.load(open(STATS))
for k, v in stats.items():
    print(f'  {k:25s} {v}')
print(f'\nTrain parquet: {os.path.getsize(TRAIN_PARQUET)/1024/1024:.0f} MB')
print(f'Val parquet:   {os.path.getsize(VAL_PARQUET)/1024/1024:.0f} MB')

## 6. Build HF Datasets (input_ids + labels)

In [ ]:
from datasets import Dataset
import pandas as pd

train_df = pd.read_parquet(TRAIN_PARQUET)
val_df = pd.read_parquet(VAL_PARQUET)
print(f'Train chunks: {len(train_df):,}')
print(f'Val chunks:   {len(val_df):,}')

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

# labels = input_ids for causal LM (Trainer shifts internally)
def add_labels(ex):
    ex['labels'] = ex['input_ids']
    return ex

train_ds = train_ds.map(add_labels, num_proc=4)
val_ds = val_ds.map(add_labels, num_proc=4)
print(f'\nFirst chunk length: {len(train_ds[0]["input_ids"])}')

## 7. Load Trendyol-7B base (bf16 + gradient checkpointing)

In [ ]:
import torch
from transformers import LlamaTokenizer, AutoModelForCausalLM

BASE = 'Trendyol/Trendyol-LLM-7b-base-v1.0'
print(f'Loading tokenizer: {BASE}')
tokenizer = LlamaTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'  vocab={tokenizer.vocab_size}, eos={tokenizer.eos_token_id}, pad={tokenizer.pad_token_id}')

print(f'\nLoading model in bf16: {BASE}')
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.gradient_checkpointing_enable()
model.config.use_cache = False  # required when gradient_checkpointing is on
print(f'  params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B')
print(f'  device: {next(model.parameters()).device}')
print(f'  dtype:  {next(model.parameters()).dtype}')

## 8. torch.load weights_only patch

PyTorch 2.6+ default'unu `weights_only=True` yaptı, eski checkpoint'lerde rng_state.pth açılamıyor. Trust ettiğimiz kendi checkpoint'lerimiz için False'a indir.

In [ ]:
import torch
_orig_load = torch.load
def _patched_load(*a, **k):
    k.setdefault('weights_only', False)
    return _orig_load(*a, **k)
torch.load = _patched_load
print('torch.load patched: weights_only=False default')

## 9. Full training (1 epoch, ~38 saat A100)

**OUT_DIR = Drive path** — checkpoint'ler persistent.

Resume mantığı: Drive'da `checkpoint-*` varsa kaldığı yerden devam. Yoksa step 0'dan başlar.

Hard kill'lerden sonra: bu notebook'u baştan (Cell 1'den) sıra ile çalıştır → bu cell'e geldiğinde Drive'daki son checkpoint'ten resume eder.

In [ ]:
import math, os, time
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

OUT_DIR = DRIVE_OUT  # /content/drive/MyDrive/trakad-7b-base
EFFECTIVE_BATCH = 32
EPOCHS = 1

steps_per_epoch = math.ceil(len(train_ds) / EFFECTIVE_BATCH)
print(f'Output dir (Drive): {OUT_DIR}')
print(f'Steps/epoch: {steps_per_epoch:,}')

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=EFFECTIVE_BATCH,
    learning_rate=1e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=True,
    logging_steps=20,
    save_strategy='steps',
    save_steps=1000,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=1000,
    report_to=[],
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)

# Drive'da checkpoint varsa resume
resume = os.path.isdir(OUT_DIR) and any(
    name.startswith('checkpoint-') for name in os.listdir(OUT_DIR)
)
print(f'Resume from checkpoint: {resume}')

t0 = time.time()
trainer.train(resume_from_checkpoint=resume)
dt = time.time() - t0
print(f'\nFull training done in {dt/3600:.2f} hours -> {OUT_DIR}')

tokenizer.save_pretrained(OUT_DIR)
print('Tokenizer saved alongside model.')

## 10. Final val PPL sanity check

In [ ]:
import math
eval_metrics = trainer.evaluate()
print(f'Final val loss: {eval_metrics["eval_loss"]:.4f}')
print(f'Final val PPL:  {math.exp(eval_metrics["eval_loss"]):.2f}')

## 11. Sample Turkish academic generations

Model'in Türkçe akademik üslupta tutarlı metin üretip üretmediğini gör.

In [ ]:
prompts = [
    'Bu çalışmada, derin öğrenme yöntemleri kullanılarak Türkçe metinler üzerinde',
    'Yenilenebilir enerji kaynaklarından rüzgar enerjisinin Türkiye\'deki potansiyeli',
    'Akademik araştırmalarda doğal dil işleme teknikleri',
]
model.eval()
for p in prompts:
    inputs = tokenizer(p, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, do_sample=True, top_p=0.9, temperature=0.7)
    print(f'\n=== {p}')
    print(tokenizer.decode(out[0], skip_special_tokens=True))

## 12. Push to HF Hub: `hakansabunis/trakad-7b-base`

In [ ]:
from huggingface_hub import HfApi

REPO = 'hakansabunis/trakad-7b-base'
api = HfApi(token=hf_token)
api.create_repo(REPO, repo_type='model', exist_ok=True, private=False, token=hf_token)

print(f'Uploading {OUT_DIR} -> {REPO}')
api.upload_folder(
    folder_path=OUT_DIR,
    repo_id=REPO,
    repo_type='model',
    commit_message='Trakad-7B-base — continued pre-train of Trendyol-LLM-7b-base-v1.0 on 387M tokens of Turkish academic abstracts',
    token=hf_token,
    allow_patterns=['*.safetensors', '*.json', '*.bin', 'tokenizer*', '*.txt', '*.model'],
)
print(f'\nDone: https://huggingface.co/{REPO}')